In [0]:
%run "../libs/libs"

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col, to_date

In [0]:
paymentdetail_df = spark.read.format("delta").table("dev.tracking_sl.paymentdetail")
dimpayment_df = spark.read.format("delta").table("dev.tracking_gd.dimpayment")
dimbank_df = spark.read.format("delta").table("dev.tracking_gd.dimbank")


In [0]:
tdetail_df = (
    paymentdetail_df.join(
        dimpayment_df,
        (paymentdetail_df.refno == dimpayment_df['payment_ref']),
        "inner"
    )  
    .join(
        dimbank_df,
        (paymentdetail_df.bankname == dimbank_df['bankname']),
        "inner"
    )     
.select(
        col('datekey'),
        col('paymentid').alias('payment_id'),
        col('bankid').alias('bank_id'),
        col('paidamount').alias('paymentamount')
    )   
)

In [0]:
tdetail_df = tdetail_df.withColumn('payer_id',lit('1')).withColumn('receiver_id',lit('1')).withColumn('paymenttype',lit('na')).withColumn('ingestion_time',current_timestamp())


                               

In [0]:
tdetail_df.createOrReplaceTempView("temp_factpayment")

In [0]:
%sql
merge into dev.tracking_gd.factpayment tgt
USING ( 
  SELECT DISTINCT * 
  FROM temp_factpayment 
) src 
on tgt.payment_id = src.payment_id
when not matched then
  insert (datekey, payment_id, paymenttype,payer_id,receiver_id,bank_id,paymentamount, ingestion_time)
  Values(src.datekey,src.payment_id,src.paymenttype,src.payer_id,src.receiver_id,src.bank_id,src.paymentamount,src.ingestion_time)